# COSC726 · Lab 0 — Engineering Baseline & Trace Literacy

**Agentic Artificial Intelligence · Week 1 · 2-hour supervised lab**

Two goals, and the second is the one that lasts:

1. **A working, reproducible setup** — environment verified, starter repo cloned, CI green on your first push
2. **Trace literacy** — read a real agent trace closely enough to audit its evidence, its permissions, and its stopping decision

The trace is **pre-recorded and deterministic**. No model, API key, or paid account is required. The `DECIDE`
entries are *authored rationale summaries* for teaching and audit — they are **not** private model reasoning.

> **Running example:** the customer-support agent helping **Layla** with order **#A1032**. It follows us all
> term, gaining one capability each week.

**Where you work:** this notebook runs in **Google Colab** (or locally, if you prefer).
**What you submit:** push this executed notebook to `week01/` in your repo, then tag `week-01-complete`.

## Part A · Environment check (~15 min)

A `WARN` is not a failure — it tells you what to repair or report **before you leave today**. Setup problems
compound silently, so today is the cheapest possible day to fix them.

In [27]:
from __future__ import annotations

import importlib.util
import platform
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

print(f"Python {platform.python_version()} on {platform.system()} ({platform.machine()})")
print("PASS · Python 3.11+" if sys.version_info >= (3, 11)
      else "WARN · install Python 3.11 or newer before Lab 1")
print(f"Environment: {'Google Colab (managed)' if IN_COLAB else 'local machine'}")
print(f"Working directory: {Path.cwd()}")

Python 3.14.3 on Windows (AMD64)
PASS · Python 3.11+
Environment: local machine
Working directory: d:\MASTER\Semester 2\Agentic Artificial Intelligence\lab0


In [28]:
# Isolation check. Colab already gives you a clean managed runtime, so a venv is
# only expected when you are running locally.
if IN_COLAB:
    print("PASS · Colab runtime is isolated and disposable — no virtualenv needed here.")
    print("       Note: Colab RESETS between sessions. GitHub is your permanent record.")
    in_venv = True          # not applicable in Colab; treated as satisfied
else:
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    if in_venv:
        print(f"PASS · virtual environment active: {sys.prefix}")
    else:
        print("WARN · no virtual environment detected")
        if platform.system() == "Windows":
            print(r"      create: python -m venv .venv    activate: .venv\Scripts\activate")
        else:
            print("      create: python3 -m venv .venv    activate: source .venv/bin/activate")

PASS · virtual environment active: d:\MASTER\Semester 2\Agentic Artificial Intelligence\lab0\.venv


In [29]:
def run_text(command: list[str]) -> str:
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return (result.stdout or result.stderr).strip()
    except OSError as exc:
        return f"ERROR: {exc}"

git_ok = shutil.which("git") is not None
if git_ok:
    print("PASS ·", run_text(["git", "--version"]))
    name = run_text(["git", "config", "user.name"])
    email = run_text(["git", "config", "user.email"])
    if name and email and not name.startswith("ERROR"):
        print(f"PASS · Git identity: {name} <{email}>")
    else:
        print("WARN · set your identity:")
        print('      git config --global user.name "Your Name"')
        print('      git config --global user.email "you@university.ac.uk"')
else:
    print("WARN · Git is not on PATH")

print("PASS · kernel executable:", sys.executable)

PASS · git version 2.50.1.windows.1
PASS · Git identity: EnasMohammed <enassmohammed93@gmail.com>
PASS · kernel executable: d:\MASTER\Semester 2\Agentic Artificial Intelligence\lab0\.venv\Scripts\python.exe


### The starter repo — you *inherit* the practices, you don't build them

Clone the classroom starter in a terminal (or a Colab cell prefixed with `!`). It already ships
`pytest`, linting, and a CI workflow **preconfigured** — your job this week is to get a green CI run, not
to assemble a toolchain from scratch.

```bash
git clone <classroom-starter-url> cosc726-<surname>
cd cosc726-<surname>
python -m pip install -r requirements.txt      # version-pinned — do NOT upgrade mid-term
python COSC726_W01_hello_agent_mock.py --self-test
```

Expected: `TRACE SELF-TEST PASSED`. That command is also what CI runs on your first push — so if it passes
locally and CI is red, the difference is your environment, and that is exactly what this week is about.

## Part B · Run the deterministic trace (~15 min)

Read it **twice**: first as a support story, then as a sequence of *observations, decisions, actions, and
state changes*. Watch especially for what the agent does **not** do.

In [30]:
MOCK_TRACE = [
    (1, "USER", "Hi, I'm Layla. Where is my order #A1032? It was meant to arrive Tuesday."),
    (2, "DECIDE", "I need the current order status. Use the read-only lookup_order tool."),
    (3, "ACT", 'lookup_order(order_id="A1032")'),
    (4, "OBSERVE", '{"order_id":"A1032","item":"desk lamp","status":"delayed at depot",'
                   '"promised_day":"Tuesday","new_eta":"Friday"}'),
    (5, "DECIDE", "The delay is three calendar days. Check the policy before mentioning compensation."),
    (6, "ACT", 'get_late_delivery_policy(topic="goodwill credit")'),
    (7, "OBSERVE", '{"eligibility":"delay of 3 or more calendar days",'
                   '"remedy":"10% goodwill credit","approval_required":true}'),
    (8, "DECIDE", "The order appears eligible. Applying a credit changes the account, so request "
                  "human approval rather than claim it happened."),
    (9, "ACT", 'request_approval(action="apply_goodwill_credit", order_id="A1032", amount_percent=10)'),
    (10, "OBSERVE", '{"approval_id":"APR-2048","status":"pending","account_changed":false}'),
    (11, "DECIDE", "I have verified the facts and opened the required approval request. Stop and "
                   "report the status, evidence, and next step."),
    (12, "RESPOND", "Hi Layla - order #A1032 is delayed at the depot and is now due Friday. Because "
                    "that is three calendar days late, it appears eligible for a 10% goodwill credit. "
                    "I submitted approval request APR-2048; no account change has been made yet."),
]

def play_trace(upto: int | None = None) -> None:
    events = MOCK_TRACE if upto is None else MOCK_TRACE[:upto]
    width = max(len(kind) for _, kind, _ in MOCK_TRACE)
    for step, kind, text in events:
        print(f"[{kind:<{width}}] step {step:>2}: {text}")
        print("-" * 108)

play_trace()

[USER   ] step  1: Hi, I'm Layla. Where is my order #A1032? It was meant to arrive Tuesday.
------------------------------------------------------------------------------------------------------------
[DECIDE ] step  2: I need the current order status. Use the read-only lookup_order tool.
------------------------------------------------------------------------------------------------------------
[ACT    ] step  3: lookup_order(order_id="A1032")
------------------------------------------------------------------------------------------------------------
[OBSERVE] step  4: {"order_id":"A1032","item":"desk lamp","status":"delayed at depot","promised_day":"Tuesday","new_eta":"Friday"}
------------------------------------------------------------------------------------------------------------
[DECIDE ] step  5: The delay is three calendar days. Check the policy before mentioning compensation.
----------------------------------------------------------------------------------------------------

**The move to notice:** at step 8 the agent decides it is *eligible* for a credit — and then, at step 9,
**does not apply it**. It opens an approval request instead, and step 10 records `"account_changed": false`.
The final message says the credit "appears eligible" and that no change has been made. That restraint is the
difference between Level 3 and Level 4 on the lecture's autonomy spectrum, and it is designed, not accidental.

## Part C · Classify the loop (~25 min)

Label **all 12 events**:

| Phase | Meaning |
|---|---|
| `sense` | receive information from the environment |
| `reason` | decide or plan the next step |
| `act` | call a tool, request approval, change state, or respond |
| `observe` | register the result of an action and update task state |

Steps **4, 7 and 10** accept either `sense` or `observe` — in software agents a tool result is often both the
outcome of the last action and the next perception. Choose one and be ready to defend it.

In [31]:

my_annotations = {
    1: "sense",
    2: "reason",
    3: "act",
    4: "observe",
    5: "reason",
    6: "act",
    7: "observe",
    8: "reason",
    9: "act",
    10: "observe",
    11: "reason",
    12: "act"
}

ANSWER_KEY = {1: ["sense"], 2: ["reason"], 3: ["act"], 4: ["observe", "sense"],
              5: ["reason"], 6: ["act"], 7: ["observe", "sense"], 8: ["reason"],
              9: ["act"], 10: ["observe", "sense"], 11: ["reason"], 12: ["act"]}

def check_annotations(answers: dict[int, str]) -> int:
    correct = 0
    for step, expected_list in ANSWER_KEY.items():
        expected = set(expected_list)
        got = str(answers.get(step, "")).strip().lower()
        ok = got in expected
        correct += int(ok)
        verdict = "OK" if ok else ("UNANSWERED" if got in {"", "..."}
                                   else "EXPECTED " + " or ".join(sorted(expected)))
        print(f"step {step:>2}: {got or '(blank)':<10} {verdict}")
    print(f"\nScore: {correct}/{len(ANSWER_KEY)}")
    return correct

check_annotations(my_annotations)

step  1: sense      OK
step  2: reason     OK
step  3: act        OK
step  4: observe    OK
step  5: reason     OK
step  6: act        OK
step  7: observe    OK
step  8: reason     OK
step  9: act        OK
step 10: observe    OK
step 11: reason     OK
step 12: act        OK

Score: 12/12


12

Do the same in `COSC726_W01_hello_agent_mock.py` (edit its `ANNOTATIONS` dict), then verify:

```bash
python COSC726_W01_hello_agent_mock.py --check      # target: 12/12
```
In Colab, prefix with `!`. Both the notebook and the script must reach 12/12.

## Part D · Evidence audit (~20 min)

Replace every `TODO`. Two or three precise sentences each; **cite step numbers** as evidence.

In [32]:
TRACE_AUDIT = """
Name: Enas Mohammed
Student ID: Not yet
--------------------------------------------------------------------------
1. EVIDENCE. Which observations support the final response? Cite step numbers.

The final response is supported by the observations in steps 4, 7, and 10. Step 4 confirms the order is delayed until Friday, step 7 shows that the order appears eligible for a 10% goodwill credit, and step 10 confirms that the approval request is pending and no account change has been made.

2. PERMISSIONS. Which tools are read-only, and which action could lead to an
   account change? Why is the approval gate appropriate?

The tools lookup_order (step 3) and get_late_delivery_policy (step 6) are read-only because they only retrieve information. The request_approval action (step 9) is related to a possible account change, so requiring human approval is appropriate to prevent unauthorized changes.

3. STATE CHANGE. What evidence shows the credit has NOT yet been applied?

Step 10 explicitly states "account_changed": false, showing that no modification has been made to the customer's account. The final response also says the credit appears eligible and that approval is still pending.

4. TERMINATION. Why is step 11 a valid stop decision? Name one condition that
   should instead cause escalation or another loop iteration.

Step 11 is a valid stopping point because the agent has collected the necessary information, submitted the approval request, and informed the customer of the next step. Another loop or escalation would be needed if the approval request failed or additional information was required.

5. TRACE LIMITS. Why should DECIDE be treated as a designed rationale summary,
   not as guaranteed access to private model reasoning?

The DECIDE entries are authored summaries created for teaching and auditing purposes. They describe the intended reasoning process but should not be interpreted as the model's private reasoning.
"""

remaining_audit = TRACE_AUDIT.count("TODO")
print(TRACE_AUDIT)
print("PASS · trace audit complete" if remaining_audit == 0
      else f"WARN · {remaining_audit} TODO item(s) remain")


Name: Enas Mohammed
Student ID: Not yet
--------------------------------------------------------------------------
1. EVIDENCE. Which observations support the final response? Cite step numbers.

The final response is supported by the observations in steps 4, 7, and 10. Step 4 confirms the order is delayed until Friday, step 7 shows that the order appears eligible for a 10% goodwill credit, and step 10 confirms that the approval request is pending and no account change has been made.

2. PERMISSIONS. Which tools are read-only, and which action could lead to an
   account change? Why is the approval gate appropriate?

The tools lookup_order (step 3) and get_late_delivery_policy (step 6) are read-only because they only retrieve information. The request_approval action (step 9) is related to a possible account change, so requiring human approval is appropriate to prevent unauthorized changes.

3. STATE CHANGE. What evidence shows the credit has NOT yet been applied?

Step 10 explicitly st

## Part E · PEAS and autonomy (~15 min)

Specify the system. Use the trace as evidence — do not answer from intuition alone. (This feeds directly
into the seminar's PEAS exercise.)

In [33]:
DESIGN_WORKSHEET = """
PEAS
----
Performance measure: Provide accurate order information, follow company policy, and avoid unauthorized account changes.

Environment: Customer support system, order database, delivery policy database, and approval workflow.

Actuators: Lookup order, retrieve policy, request approval, and respond to the customer.

Sensors: Customer request, order lookup result, policy lookup result, and approval status.

AUTONOMY AND RISK
-----------------
Starting autonomy level (1-4): 3

Evidence from the trace (cite step numbers): Steps 8, 9, and 10 show that the agent requests approval instead of changing the account directly.

Highest-risk action in the trace: Requesting approval to apply the goodwill credit (step 9).

What evidence would justify moving up one level? Evidence that the agent can safely apply account changes automatically with reliable safeguards and policy compliance.
"""

remaining_peas = DESIGN_WORKSHEET.count("TODO")
print(DESIGN_WORKSHEET)
print("PASS · design worksheet complete" if remaining_peas == 0
      else f"WARN · {remaining_peas} TODO item(s) remain")


PEAS
----
Performance measure: Provide accurate order information, follow company policy, and avoid unauthorized account changes.

Environment: Customer support system, order database, delivery policy database, and approval workflow.

Actuators: Lookup order, retrieve policy, request approval, and respond to the customer.

Sensors: Customer request, order lookup result, policy lookup result, and approval status.

AUTONOMY AND RISK
-----------------
Starting autonomy level (1-4): 3

Evidence from the trace (cite step numbers): Steps 8, 9, and 10 show that the agent requests approval instead of changing the account directly.

Highest-risk action in the trace: Requesting approval to apply the goodwill credit (step 9).

What evidence would justify moving up one level? Evidence that the agent can safely apply account changes automatically with reliable safeguards and policy compliance.

PASS · design worksheet complete


## Part F · Decision memo (~20 min)

**Every lab in this module closes with the same half-page memo.** It is formative — it carries no direct
marks — but it is the postgraduate standard, and it feeds your project report and the exam's design question.

This week the sharpest question is **reproducibility**: *could a marker clone your repository and re-run your
work unchanged?*

In [34]:
DECISION_MEMO = """
COSC726 · Lab 0 decision memo
Name: Enas Mohammed
--------------------------------------------------------------------------
1. BETTER THAN WHAT? What is the baseline this agent should be compared with
   (e.g. a fixed auto-reply template)? What does the trace do that the baseline
   cannot?

A suitable baseline is a fixed auto-reply template. The agent is better because it looks up the order status, checks the late delivery policy, requests approval when needed, and produces a response based on the retrieved evidence instead of sending the same reply to every customer.

2. MEASURED BY WHAT? Propose ONE measurable criterion for whether this agent
   handled Layla well. State how you would compute it.

One measurable criterion is factual accuracy. This can be computed by comparing the statements in the final response with the information returned by the tools and calculating the percentage of correct statements.

3. AT WHAT COST? Count the tool calls and events. If each step involved a model
   call over a growing transcript, where would the cost grow fastest?

The trace contains 12 events and 3 tool-related actions (lookup_order, get_late_delivery_policy, and request_approval). The computational cost would grow fastest in the later steps because the model would need to process an increasingly longer conversation history.

4. UNDER WHAT FAILURE CONDITIONS? Pick one step and describe what the agent
   should do if that tool returned an error or a timeout instead.

If lookup_order at step 3 returned an error or timed out, the agent should report that it could not retrieve the order information, retry if appropriate, or escalate the issue to a human operator instead of guessing the status.

5. HOW REPRODUCIBLE? Could a marker clone your repo and reproduce today's
   output exactly? Name the two things most likely to break that.

Yes, because this lab uses a deterministic mock trace. The two things most likely to break reproducibility are missing project files (or incorrect repository contents) and differences in the local software environment, such as Python version or missing dependencies.

6. WHAT ENGINEERING DECISION FOLLOWS? State one concrete change you would make
   before this agent went anywhere near a real customer.

Before deploying this agent, I would add stronger error handling and logging for every tool call to ensure failures are detected, recorded, and handled safely.
"""

remaining_memo = DECISION_MEMO.count("TODO")
print(DECISION_MEMO)
print("PASS · decision memo complete" if remaining_memo == 0
      else f"WARN · {remaining_memo} TODO item(s) remain")


COSC726 · Lab 0 decision memo
Name: Enas Mohammed
--------------------------------------------------------------------------
1. BETTER THAN WHAT? What is the baseline this agent should be compared with
   (e.g. a fixed auto-reply template)? What does the trace do that the baseline
   cannot?

A suitable baseline is a fixed auto-reply template. The agent is better because it looks up the order status, checks the late delivery policy, requests approval when needed, and produces a response based on the retrieved evidence instead of sending the same reply to every customer.

2. MEASURED BY WHAT? Propose ONE measurable criterion for whether this agent
   handled Layla well. State how you would compute it.

One measurable criterion is factual accuracy. This can be computed by comparing the statements in the final response with the information returned by the tools and calculating the percentage of correct statements.

3. AT WHAT COST? Count the tool calls and events. If each step involved a

## Part G · Submission readiness

Run this after completing Parts C–F.

In [35]:
score = check_annotations(my_annotations)

checks = {
    "Python 3.11+":               sys.version_info >= (3, 11),
    "Isolated environment":       in_venv,
    "Git available":              git_ok,
    "Annotations 12/12":          score == 12,
    "Trace audit complete":       TRACE_AUDIT.count("TODO") == 0,
    "PEAS / autonomy complete":   DESIGN_WORKSHEET.count("TODO") == 0,
    "Decision memo complete":     DECISION_MEMO.count("TODO") == 0,
}

print()
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'WARN'} · {label}")

print()
if all(checks.values()):
    print("READY TO SUBMIT — now follow the push steps in the next cell.")
else:
    print("Resolve the warnings above, or document the setup issue with a demonstrator.")

step  1: sense      OK
step  2: reason     OK
step  3: act        OK
step  4: observe    OK
step  5: reason     OK
step  6: act        OK
step  7: observe    OK
step  8: reason     OK
step  9: act        OK
step 10: observe    OK
step 11: reason     OK
step 12: act        OK

Score: 12/12

PASS · Python 3.11+
PASS · Isolated environment
PASS · Git available
PASS · Annotations 12/12
PASS · Trace audit complete
PASS · PEAS / autonomy complete
PASS · Decision memo complete

READY TO SUBMIT — now follow the push steps in the next cell.


### Push it (this is the submission)

1. **Restart & Run All** so the notebook is clean top-to-bottom.
2. **From Colab:** `File → Save a copy in GitHub` → repo `cosc726-<surname>`, path **`week01/lab0.ipynb`**,
   and write a real commit message (*"Lab 0: trace annotated 12/12, memo complete"* — not *"update"*).
3. **Also push** your edited `COSC726_W01_hello_agent_mock.py` to `week01/`.
4. **Tag the checkpoint:** on GitHub, *Releases → new tag* **`week-01-complete`**. This is your safety net —
   if a later week breaks, you can branch cleanly from here.
5. **Check CI is green** on the push. A red CI is a finding, not a disaster — but report it today.

**Deadline:** pushed before the Week 2 lecture. The teaching team reads your repository directly; there is no
separate upload. Your repo link belongs in the Moodle *Repository link* box (once, in Week 1).

> **Never commit secrets.** Nothing this week needs an API key — but the habit starts now: real keys go in a
> git-ignored `.env`, and only `.env.example` is committed.

## Stretch · Predict, then structure

1. Run `play_trace(upto=4)` — stop right after the first tool result — and **write down your prediction** of
   steps 5–12 before revealing them. Did you predict the *approval gate*, or did you expect the agent to
   apply the credit itself?
2. Run `!python COSC726_W01_hello_agent_mock.py --json` and look at the trace as structured data.
3. Propose **four fields** an observability system should add — for example timestamp, run ID, actor, tool
   latency, permission class, state delta, or stop reason. You will meet the real version of this list in
   Week 2 (telemetry) and wire it into CI in Week 11.

---
**Next week:** LLM foundations from zero — tokens, context budgets, message roles, sampling, and the
difference between model output and system behaviour.